输入一张大小为 $3 \times 32 \times 32$（通道数 × 高 × 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$。设定填充（Padding）为 2，步幅（Stride）为 2。

1. 请计算该卷积层输出的特征图（Feature Map）的尺寸（通道数 × 高 × 宽）。
2. 计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行多少次点乘（乘法）操作？

### 问题1：计算输出特征图尺寸
卷积层输出特征图的高/宽计算公式为：
$$
H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2 \times P - K}{S} \right\rfloor + 1
$$
$$
W_{\text{out}} = \left\lfloor \frac{W_{\text{in}} + 2 \times P - K}{S} \right\rfloor + 1
$$
其中：
- 输入尺寸 $H_{\text{in}} = W_{\text{in}} = 32$
- 卷积核尺寸 $K = 5$
- 填充 $P = 2$
- 步幅 $S = 2$
- 输出通道数等于卷积核数量，即 $C_{\text{out}} = 16$

代入计算：
$$
H_{\text{out}} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
$$
同理，$W_{\text{out}} = 16$。

因此，输出特征图尺寸为：
$$
16 \times 16 \times 16
$$
（通道数 × 高 × 宽）

---

### 问题2：计算单个像素的点乘次数
单个输出通道的一个像素值，由一个卷积核与输入特征图的局部区域进行点乘得到。卷积核的尺寸为 $3 \times 5 \times 5$（输入通道数 × 卷积核高 × 卷积核宽），因此点乘（乘法）操作次数为：
$$
3 \times 5 \times 5 = 75
$$

不使用深度学习框架的底层 Pooling API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。

In [5]:
import numpy as np

def max_pool2d(input_tensor, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    参数:
        input_tensor: 输入张量，形状为 (N, C, H, W)
        kernel_size: 池化核大小，单个整数或 (k_h, k_w)
        stride: 步幅，单个整数或 (s_h, s_w)，默认等于kernel_size
        padding: 填充大小，单个整数或 (p_h, p_w)
    返回:
        output_tensor: 池化后的张量，形状为 (N, C, H_out, W_out)
    """
    # 参数标准化
    if isinstance(kernel_size, int):
        k_h, k_w = kernel_size, kernel_size
    else:
        k_h, k_w = kernel_size
    if stride is None:
        s_h, s_w = k_h, k_w
    elif isinstance(stride, int):
        s_h, s_w = stride, stride
    else:
        s_h, s_w = stride
    if isinstance(padding, int):
        p_h, p_w = padding, padding
    else:
        p_h, p_w = padding
    
    # 获取输入形状
    N, C, H_in, W_in = input_tensor.shape
    
    # 计算输出尺寸
    H_out = (H_in + 2 * p_h - k_h) // s_h + 1
    W_out = (W_in + 2 * p_w - k_w) // s_w + 1
    
    # 填充输入（填充为负无穷，避免0干扰最大值计算）
    padded_input = np.pad(
        input_tensor, 
        ((0, 0), (0, 0), (p_h, p_h), (p_w, p_w)), 
        mode='constant', 
        constant_values=-np.inf
    )
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 遍历池化窗口
    for i in range(H_out):
        for j in range(W_out):
            # 计算当前窗口的位置
            h_start = i * s_h
            h_end = h_start + k_h
            w_start = j * s_w
            w_end = w_start + k_w
            # 取窗口区域的最大值
            window = padded_input[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2, 3))
    
    return output

# 测试示例
if __name__ == "__main__":
    # 生成随机输入 (N=1, C=3, H=4, W=4)
    x = np.random.randn(1, 3, 4, 4)
    # 池化核大小2x2，步幅2，填充0
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("输入形状:", x.shape)
    print("输出形状:", out.shape)
    print("输出结果:\n", out)

输入形状: (1, 3, 4, 4)
输出形状: (1, 3, 2, 2)
输出结果:
 [[[[ 0.96876619  0.47890524]
   [ 0.82412329  1.62633845]]

  [[ 1.92903133  1.04222159]
   [ 0.40530589 -0.03350999]]

  [[ 0.62082912  2.16636447]
   [ 1.36349335  1.22925266]]]]


在 VGG 网络中，作者频繁使用多个 $3 \times 3$ 卷积核级联来代替较大的卷积核（如 $5 \times 5$ 或 $7 \times 7$）。假设输入和输出的特征图通道数均为 $C$。

1. 计算一个 $5 \times 5$ 卷积层（不带偏置）的参数量。
2. 计算两个串联的 $3 \times 3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量。

### 问题1：$5 \times 5$ 卷积层参数量
不带偏置的卷积层参数量公式为：
$$
\text{参数量} = \text{输入通道数} \times \text{输出通道数} \times \text{卷积核高} \times \text{卷积核宽}
$$
已知输入/输出通道数均为 $C$，卷积核大小为 $5 \times 5$，代入公式：
$$
\text{参数量}_{5\times5} = C \times C \times 5 \times 5 = 25C^2
$$

---

### 问题2：两个串联的 $3 \times 3$ 卷积层总参数量
每个不带偏置的 $3 \times 3$ 卷积层参数量为：
$$
\text{单层参数量}_{3\times3} = C \times C \times 3 \times 3 = 9C^2
$$
两个串联的卷积层总参数量为两层参数量之和：
$$
\text{总参数量} = 9C^2 + 9C^2 = 18C^2
$$

NiN 网络的核心创新是引入了 “1x1 卷积” 组成的 NiN 块来代替传统的全连接层，以减少参数量。请使用 PyTorch（`torch.nn.Sequential`）定义一个标准的 NiN 块（NiN Block）。

要求：NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通的卷积层（指定窗口大小 `kernel_size`，步幅 `stride`，填充 `padding`）以及两个随后的 $1 \times 1$ 卷积层级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。

In [6]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.nin_block = nn.Sequential(
            # 第一层：普通卷积 + ReLU
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.ReLU(),
            # 第二层：1x1卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(),
            # 第三层：1x1卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU()
        )

    def forward(self, x):
        return self.nin_block(x)

# 测试示例
if __name__ == "__main__":
    # 定义一个NiN块：输入通道3，输出通道16，卷积核3x3，步幅1，填充1
    nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    # 随机输入 (batch_size=2, channels=3, height=32, width=32)
    x = torch.randn(2, 3, 32, 32)
    output = nin_block(x)
    print("输入形状:", x.shape)
    print("输出形状:", output.shape)

输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 16, 32, 32])


在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$。假设当前批量归一化层学到的缩放参数 $\gamma = 2$，平移参数 $\beta = 1$，常数 $\epsilon = 0$。

请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 $y_1, y_2, y_3, y_4$。

### 步骤1：计算小批量均值 $\mu$
均值公式：
$$
\mu = \frac{1}{N} \sum_{i=1}^N x_i
$$
其中 $N=4$，代入数据：
$$
\mu = \frac{2 + 4 + 6 + 8}{4} = \frac{20}{4} = 5
$$

### 步骤2：计算小批量方差 $\sigma^2$
方差公式：
$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2
$$
代入数据：
$$
\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = 5
$$

### 步骤3：标准化处理
标准化公式：
$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
已知 $\epsilon=0$，则：
$$
\hat{x}_1 = \frac{2-5}{\sqrt{5}} = -\frac{3}{\sqrt{5}}
$$
$$
\hat{x}_2 = \frac{4-5}{\sqrt{5}} = -\frac{1}{\sqrt{5}}
$$
$$
\hat{x}_3 = \frac{6-5}{\sqrt{5}} = \frac{1}{\sqrt{5}}
$$
$$
\hat{x}_4 = \frac{8-5}{\sqrt{5}} = \frac{3}{\sqrt{5}}
$$

### 步骤4：缩放与平移
最终输出公式：
$$
y_i = \gamma \cdot \hat{x}_i + \beta
$$
已知 $\gamma=2$，$\beta=1$，代入得：
$$
y_1 = 2 \times \left(-\frac{3}{\sqrt{5}}\right) + 1 = 1 - \frac{6}{\sqrt{5}}
$$
$$
y_2 = 2 \times \left(-\frac{1}{\sqrt{5}}\right) + 1 = 1 - \frac{2}{\sqrt{5}}
$$
$$
y_3 = 2 \times \frac{1}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}}
$$
$$
y_4 = 2 \times \frac{3}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}}
$$

有理化分母后，结果为：
$$
y_1 = 1 - \frac{6\sqrt{5}}{5}, \quad y_2 = 1 - \frac{2\sqrt{5}}{5}, \quad y_3 = 1 + \frac{2\sqrt{5}}{5}, \quad y_4 = 1 + \frac{6\sqrt{5}}{5}
$$

残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

要求：该块包含两个具有相同输出通道数的 $3 \times 3$ 卷积层，每个卷积层后跟一个批量归一化层。如果 `use_1x1conv=True`，则需要对输入应用一个 $1 \times 1$ 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输出进行按元素相加（$f(x) + x$）。

In [7]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        # 主路径：两个3x3卷积层 + 批量归一化
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 残差路径：1x1卷积（可选），用于调整通道和形状
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        else:
            self.conv3 = None
        
    def forward(self, x):
        # 主路径前向传播
        y = self.conv1(x)
        y = self.bn1(y)
        y = nn.ReLU()(y)
        y = self.conv2(y)
        y = self.bn2(y)
        
        # 残差路径前向传播（调整输入维度）
        if self.conv3 is not None:
            x = self.conv3(x)
        
        # 残差连接 + ReLU激活
        return nn.ReLU()(y + x)

# 测试示例
if __name__ == "__main__":
    # 场景1：输入输出通道相同，无需1x1卷积（基础残差块）
    residual1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x1 = torch.randn(1, 64, 32, 32)
    out1 = residual1(x1)
    print("场景1 - 输入形状:", x1.shape, "输出形状:", out1.shape)
    
    # 场景2：输入输出通道不同，步幅为2，使用1x1卷积调整（下采样残差块）
    residual2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
    x2 = torch.randn(1, 64, 32, 32)
    out2 = residual2(x2)
    print("场景2 - 输入形状:", x2.shape, "输出形状:", out2.shape)

场景1 - 输入形状: torch.Size([1, 64, 32, 32]) 输出形状: torch.Size([1, 64, 32, 32])
场景2 - 输入形状: torch.Size([1, 64, 32, 32]) 输出形状: torch.Size([1, 128, 16, 16])


在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。请回答以下关于微调理论的问题：
1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

### 问题1解答
- **底层特征提取层**：
  预训练模型的底层（如卷积层）已经学习到了通用的、可迁移的图像特征（如边缘、纹理、基础形状等），这些特征对大多数计算机视觉任务都具有普适性。如果设置较大的学习率，会大幅修改这些已学到的稳定特征，破坏预训练模型的有效知识，导致模型“遗忘”通用特征，反而降低泛化能力。因此，设置较小的学习率（或直接冻结参数）可以保留这些通用特征，仅做微小调整以适配目标数据分布。

- **顶层输出层**：
  顶层输出层是为目标任务新初始化的，参数是随机的，完全不具备目标任务的知识。需要较大的学习率来快速学习目标任务的特定规则（如分类边界），实现快速收敛。同时，顶层输出层的任务特异性强，与预训练模型的通用特征关联度低，较大的学习率不会影响底层的通用特征提取能力，因此可以设置较高的学习率。

---

### 问题2解答
当目标数据集非常小且与源数据集高度相似时，模型极易过拟合到少量目标数据，应采取以下策略：
1. **冻结底层，仅训练顶层**：
   完全冻结预训练模型的所有底层特征提取层，仅训练新初始化的顶层输出层。底层完全保留预训练学到的通用特征，不进行任何更新，避免因少量数据修改通用特征导致过拟合。
2. **极低学习率微调（可选）**：
   若需要微调，仅解冻顶层的1-2个卷积块，并设置极低的学习率（如$10^{-5}$或更小），避免大幅修改预训练参数。
3. **增加正则化**：
   在顶层输出层加入Dropout层、L2权重衰减（Weight Decay），或使用早停（Early Stopping）防止过拟合。
4. **数据增强**：
   对目标数据集进行随机裁剪、翻转、颜色扰动等数据增强操作，扩充有效数据量，减少过拟合风险。
5. **半监督/自监督辅助**：
   若有少量无标注数据，可结合半监督学习方法，利用无标注数据辅助模型学习更鲁棒的特征。

图像增广能有效增强模型的泛化能力。请利用 `torchvision.transforms` 模块创建一个组合图像增广管道（Pipeline）。
1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后的图像缩放到 $224 \times 224$ 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。

In [8]:
from torchvision import transforms
import numpy as np
from PIL import Image

# 定义图像增广管道（核心部分不变）
train_transform = transforms.Compose([
    # 1. 随机裁剪，面积比例0.08~1.0，缩放到224x224
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.08, 1.0)
    ),
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机调整亮度、对比度、饱和度，变化范围0.5
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5,
        hue=0.0
    ),
    # 4. 转换为PyTorch张量
    transforms.ToTensor()
])

# 测试示例（无需外部文件）
if __name__ == "__main__":
    # 生成一个虚拟的RGB图像（256x256像素）
    dummy_img = Image.fromarray(np.uint8(np.random.randint(0, 255, size=(256, 256, 3))))
    
    # 应用增广管道
    augmented_img = train_transform(dummy_img)
    
    # 打印结果
    print("增广后图像张量形状:", augmented_img.shape)
    print("张量数据类型:", augmented_img.dtype)

增广后图像张量形状: torch.Size([3, 224, 224])
张量数据类型: torch.float32


在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知图像中两个边界框（以「左上角 x, 左上角 y, 右下角 x, 右下角 y」格式表示）：
1. 真实框（Ground Truth） $A = [10, 10, 50, 50]$
2. 预测框（Prediction Box） $B = [30, 30, 70, 70]$

请计算边界框 $A$ 和边界框 $B$ 之间的 IoU 准确值。

### 步骤1：计算单个边界框的面积
- 真实框 $A$ 的宽高：
  $w_A = x_{A2} - x_{A1} = 50 - 10 = 40$，$h_A = y_{A2} - y_{A1} = 50 - 10 = 40$
  面积 $S_A = w_A \times h_A = 40 \times 40 = 1600$

- 预测框 $B$ 的宽高：
  $w_B = x_{B2} - x_{B1} = 70 - 30 = 40$，$h_B = y_{B2} - y_{B1} = 70 - 30 = 40$
  面积 $S_B = w_B \times h_B = 40 \times 40 = 1600$

### 步骤2：计算交集区域的坐标与面积
交集区域的边界由两个框的坐标共同决定：
- 交集左上角：$x_{\text{inter1}} = \max(x_{A1}, x_{B1}) = \max(10, 30) = 30$，$y_{\text{inter1}} = \max(y_{A1}, y_{B1}) = \max(10, 30) = 30$
- 交集右下角：$x_{\text{inter2}} = \min(x_{A2}, x_{B2}) = \min(50, 70) = 50$，$y_{\text{inter2}} = \min(y_{A2}, y_{B2}) = \min(50, 70) = 50$

交集区域的宽高：
$w_{\text{inter}} = x_{\text{inter2}} - x_{\text{inter1}} = 50 - 30 = 20$
$h_{\text{inter}} = y_{\text{inter2}} - y_{\text{inter1}} = 50 - 30 = 20$

交集面积：
$$
S_{\text{inter}} = w_{\text{inter}} \times h_{\text{inter}} = 20 \times 20 = 400
$$

### 步骤3：计算并集面积
并集面积公式：
$$
S_{\text{union}} = S_A + S_B - S_{\text{inter}}
$$
代入数据：
$$
S_{\text{union}} = 1600 + 1600 - 400 = 2800
$$

### 步骤4：计算IoU
IoU公式：
$$
\text{IoU} = \frac{S_{\text{inter}}}{S_{\text{union}}}
$$
代入数据：
$$
\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429
$$

在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（One-hot），若设置平滑因子 $\epsilon = 0.1$，则对于 $K$ 分类问题，真实标签对应的目标概率从 $1$ 变为 $1-\epsilon$，其余错误类别的概率从 $0$ 变为 $\frac{\epsilon}{K-1}$。

请实现一个计算标签平滑后交叉熵损失的函数。

In [9]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='mean'):
    """
    计算标签平滑后的交叉熵损失
    参数:
        logits: 模型输出的原始logits，形状为 (batch_size, num_classes)
        labels: 真实标签，形状为 (batch_size,)，值为0~num_classes-1的整数
        epsilon: 标签平滑因子，默认0.1
        reduction: 损失计算方式，可选'mean'/'sum'/'none'，默认'mean'
    返回:
        标签平滑后的交叉熵损失
    """
    num_classes = logits.size(1)
    batch_size = logits.size(0)
    
    # 步骤1：对logits进行softmax
    probs = F.softmax(logits, dim=1)
    
    # 步骤2：生成标签平滑后的目标分布
    # 先创建独热标签
    one_hot = torch.zeros_like(probs).scatter(1, labels.unsqueeze(1), 1)
    # 应用标签平滑
    smooth_labels = one_hot * (1 - epsilon) + (1 - one_hot) * (epsilon / (num_classes - 1))
    
    # 步骤3：计算交叉熵损失（负对数概率的期望）
    # 交叉熵公式：-sum(p_i * log(q_i))，其中p_i是平滑后的标签，q_i是模型输出概率
    log_probs = torch.log(probs + 1e-10)  # 加1e-10防止log(0)
    loss = -torch.sum(smooth_labels * log_probs, dim=1)
    
    # 步骤4：应用reduction
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:  # 'none'
        return loss

# 测试示例
if __name__ == "__main__":
    # 模拟输入：batch_size=2，num_classes=3
    logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.0, 0.3]])
    labels = torch.tensor([0, 1])  # 真实标签
    # 计算损失
    loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    print("标签平滑交叉熵损失:", loss.item())

标签平滑交叉熵损失: 0.5313231348991394
